# 14 端到端演示与面试题库

**用途：** 用同一问题比较手写workflow和LangGraph，并形成最终验收与面试表达。

> 使用方式：按顺序运行。出现 `PASS` 才代表本节验收成功；断言失败时先阅读紧邻的“失败定位”。默认不调用真实模型、不写生产数据库。

In [1]:
from pathlib import Path
import importlib.util
import json
import os
import sys
import tempfile

cwd = Path.cwd().resolve()
DAY1_ROOT = None
PROJECT2_ROOT = None
for candidate in [cwd, *cwd.parents]:
    if (candidate / "project2" / "agent_graph.py").exists():
        DAY1_ROOT = candidate
        PROJECT2_ROOT = candidate / "project2"
        break
    if (candidate / "agent_graph.py").exists() and (candidate / "tests").exists():
        PROJECT2_ROOT = candidate
        DAY1_ROOT = candidate.parent
        break
assert DAY1_ROOT is not None and PROJECT2_ROOT is not None, "找不到 day1/project2 项目根目录"
NOTEBOOK_ROOT = PROJECT2_ROOT / "notebooks"
for path in [str(DAY1_ROOT), str(PROJECT2_ROOT), str(NOTEBOOK_ROOT)]:
    if path not in sys.path:
        sys.path.insert(0, path)

from notebook_utils import (
    check,
    check_equal,
    file_inventory,
    load_jsonl,
    masked_environment,
    run_command,
    run_unittest,
    show_markdown,
    show_table,
    source_excerpt,
)

RUN_LIVE_MODEL_TESTS = os.getenv("RUN_LIVE_MODEL_TESTS", "0") == "1"
print(f"Python: {sys.executable}")
print(f"DAY1_ROOT: {DAY1_ROOT}")
print(f"PROJECT2_ROOT: {PROJECT2_ROOT}")
print(f"RUN_LIVE_MODEL_TESTS: {RUN_LIVE_MODEL_TESTS}")

Python: D:\new things\项目1\day1\.venv\Scripts\python.exe
DAY1_ROOT: D:\new things\项目1\day1
PROJECT2_ROOT: D:\new things\项目1\day1\project2
RUN_LIVE_MODEL_TESTS: False


In [2]:
from pathlib import Path
from agent_workflow import run_agent
import agent_graph
from handoff_repository import HandoffRepository
from memory_repository import MemoryRepository

question = "小松PC200原厂液压泵要1件，有没有现货，多少钱，发到贵阳要多久？"
baseline = run_agent(question)
temp_dir = tempfile.TemporaryDirectory()
root = Path(temp_dir.name)
saver = agent_graph.create_sqlite_checkpointer(root / "checkpoint.sqlite3")
graph = agent_graph.build_graph(
    saver,
    HandoffRepository(root / "handoff.sqlite3"),
    MemoryRepository(root / "memory.sqlite3"),
)
graph_result = agent_graph.start_graph_agent(
    question,
    thread_id="e2e-thread",
    customer_id="e2e-customer",
    approval_mode="auto",
    parser_mode="rules",
    graph=graph,
)
show_table([
    {"实现": "if-else baseline", "状态": baseline["status"], "工具": "、".join(baseline["called_tools"]), "轨迹": len(baseline.get("execution_trace", []))},
    {"实现": "LangGraph", "状态": graph_result["status"], "工具": "、".join(graph_result["called_tools"]), "轨迹": len(graph_result.get("execution_trace", []))},
])
check_equal("两套实现工具选择一致", graph_result["called_tools"], baseline["called_tools"])
check_equal("LangGraph端到端完成", graph_result["status"], "completed")
saver.conn.close()
temp_dir.cleanup()

D:\new things\项目1\day1\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,实现,状态,工具,轨迹
0,if-else baseline,completed,inventory_tool、quote_tool、logistics_tool,5
1,LangGraph,completed,inventory_tool、quote_tool、logistics_tool,11


[PASS] 两套实现工具选择一致 | actual=['inventory_tool', 'quote_tool', 'logistics_tool'], expected=['inventory_tool', 'quote_tool', 'logistics_tool']
[PASS] LangGraph端到端完成 | actual='completed', expected='completed'


In [3]:
acceptance = [
    {"模块": "RAG组件", "证据": "4条组件测试 + Top-K报告", "成功": True},
    {"模块": "workflow", "证据": "30/30", "成功": True},
    {"模块": "LangGraph", "证据": "30/30 + runtime 6/6", "成功": True},
    {"模块": "上下文/记忆", "证据": "7/7", "成功": True},
    {"模块": "多会话", "证据": "6/6 + UI切换测试", "成功": True},
    {"模块": "人工接管", "证据": "6/6 + 策略9/9", "成功": True},
    {"模块": "Harness", "证据": "12/12", "成功": True},
    {"模块": "多模态MVP", "证据": "运行时10/10", "成功": True},
    {"模块": "40张字段准确率", "证据": "等待双人gold", "成功": False},
    {"模块": "网页搜索/FastAPI/LangSmith/Multi-Agent", "证据": "尚未实现", "成功": False},
]
show_table(acceptance)
check("已实现模块都有自动证据", all(row["成功"] for row in acceptance[:8]))
check("未完成项没有被误报", not any(row["成功"] for row in acceptance[8:]))

,模块,证据,成功
0,RAG组件,4条组件测试 + Top-K报告,True
1,workflow,30/30,True
2,LangGraph,30/30 + runtime 6/6,True
3,上下文/记忆,7/7,True
4,多会话,6/6 + UI切换测试,True
5,人工接管,6/6 + 策略9/9,True
6,Harness,12/12,True
7,多模态MVP,运行时10/10,True
8,40张字段准确率,等待双人gold,False
9,网页搜索/FastAPI/LangSmith/Multi-Agent,尚未实现,False


[PASS] 已实现模块都有自动证据
[PASS] 未完成项没有被误报


{'检查项': '未完成项没有被误报', '状态': 'PASS', '说明': ''}

## 60秒项目介绍

我做了一个面向挖机配件销售的多工具Agent。项目一先完成企业知识库RAG，项目二把客服从“回答问题”升级成“按流程办事”。系统使用LangGraph维护State、条件路由、checkpoint和两类人工中断；LangChain负责模型、Prompt、结构化输出、Retriever和StructuredTool；库存、报价、物流和售后由确定性工具执行。上下文按Token预算压缩，客户事实单独治理；图片由独立视觉模型提取候选证据，必须经过质量门控和客户确认。系统具备执行轨迹、Harness、人工接管、多会话恢复和67条运行时测试。

## 高频面试题总复盘

1. Chunk size/overlap为什么是500/80，如何评测？
2. 为什么选择bge-small-zh-v1.5，中文Embedding有什么限制？
3. Top-K为什么比较1/3/5/8，K=5零失败为何不一定线上用5？
4. 检索不到、同义词、库外问题、幻觉和来源引用如何处理？
5. Chroma规模变大后怎么办？如何增量更新？
6. LangChain具体用了哪些模块？换Embedding或Vector DB改哪里？
7. 为什么LangGraph而不是纯if-else或通用Agent？
8. State、node、conditional edge、checkpoint和interrupt分别是什么？
9. 图失败如何恢复，哪些工具需要审批？
10. messages如何转换？默认能记忆多少轮？
11. 短期记忆、长期记忆、RAG和日志为什么必须分开？
12. 多会话如何恢复并防止跨客户串线？
13. AI无法处理时如何转人工并恢复原线程？
14. Harness解决哪些模型调用问题？还缺哪些生产能力？
15. 为什么文本继续DeepSeek，视觉单独使用智谱？
16. 图片为什么必须Schema、拒识、确认和人工接管？
17. 40张预跑为什么不是准确率？双人gold怎么做？
18. Streamlit、FastAPI和LangSmith分别解决什么？
19. 为什么Skills、sub-agent和multi-agent暂时后置？
20. 当前项目最诚实的生产边界是什么？

## 总复盘参考答案

1. **500/80怎么选？** 500个中文字符通常能保留一条完整业务规则，80用于缓解边界截断；它只是经当前30条业务集和9条RAG专项得到的起点，后续应联合比较检索召回、来源正确率、噪声、Token和延迟。配置在`settings.py`，切分在`build_index.py`。
2. **为什么用bge-small-zh-v1.5？** 它对中文语义友好、可在CPU运行，模型体积和延迟适合作品集。限制是专业件号、数字和表格精确匹配不稳定，所以要结合关键词、元数据、同义词和困难负例评测。
3. **Top-K为什么比较1/3/5/8？** K=1用于观察漏召回，3/5是常用候选规模，8用于观察噪声和成本上升。专项中K=5零失败只说明这9条用例，线上仍应扩大数据，并可“召回5条、阈值/Rerank后送3条”。
4. **检索失败和幻觉怎么处理？** 先判断知识缺失、切分、表达差异还是阈值问题，再用查询改写、同义词、混合检索或Rerank；证据不足时明确拒答/追问/转人工。企业知识答案返回来源，库存和价格只引用工具结果。
5. **Chroma和增量更新怎么办？** Chroma适合单实例作品集；多租户、大规模生产可迁Milvus、pgvector或托管库。文档新增或变化时按文件hash删除旧chunk并写入新chunk；Embedding或切分配置变化由fingerprint检测并触发全量重建。
6. **LangChain用了什么，怎么替换组件？** 使用Document、Splitter、HuggingFaceEmbeddings、Chroma/Retriever、Prompt、ChatOpenAI、structured output、StructuredTool和HumanMessage。替换Embedding/Vector DB集中在`rag_components.py`和构建入口，但必须重建索引并重新标定距离与Top-K。
7. **为什么用LangGraph？** 纯if-else仍作为简单回归基线，但主链需要State、动态路由、SQLite checkpoint、两类interrupt和跨进程恢复；通用Agent的模型循环又难以保证报价和售后审批，所以选择显式业务图。
8. **五个LangGraph概念是什么？** State是共享业务数据；node执行单一阶段；conditional edge按State选下一步；checkpoint按`thread_id`持久化每步状态；interrupt在审批、图片确认或人工回复处安全暂停，`Command(resume=...)`继续。
9. **失败如何恢复，哪些要审批？** 临时工具错误有限重试，checkpoint使进程重启后可用同一线程继续，幂等键避免重复动作；报价和售后工单在手动模式下审批，只读工具一般不逐次审批，但低置信或失败可转人工。
10. **messages怎么转换，能记忆多少轮？** 输入先归一化role、限制长度并记录轮次/请求ID，较早内容压成摘要，模型调用时再拼装受控Prompt或LangChain消息。默认保留8条近期消息，约4轮完整问答；会话总轮数无硬上限，但旧消息不会永久逐字进入上下文。
11. **四类记忆为什么分开？** 短期记忆服务同线程指代，长期记忆只保存跨线程白名单客户事实，RAG是企业知识，日志是审计记录。混在一起会造成客户串线、上下文膨胀、错误事实固化和隐私风险。
12. **多会话如何恢复和隔离？** `conversation_threads.sqlite3`管理目录，LangGraph checkpoint保存State。打开旧会话时目录层按`customer_id`授权，加载后再校验State归属，形成双层防护；生产还需Postgres和租户鉴权。
13. **怎样转人工并恢复？** 确定性策略在明确要求人工、重复缺信息、售后/诊断、高风险或工具失败时创建SQLite服务单并interrupt。客服领取后通过同一`thread_id`和`resume_handoff_agent`写回回复，异步渠道再经幂等outbox发送。
14. **Harness做了什么，还缺什么？** 已统一超时、有限重试、错误分类、同步并发、调用/Token/估算费用预算、结构化输出重试和脱敏日志。还缺跨进程限流、分布式熔断、Provider健康探测/自动降级和真实账单回传。
15. **为什么DeepSeek加独立智谱视觉？** 文本链已经稳定且成本较低，视觉只在图片请求触发。ModelRouter按能力分开Provider，可以独立配置、评测和降级，不需要承担全量换模风险。
16. **图片为什么需要四层控制？** Pydantic Schema限制输出形状，质量门控拒绝模糊/反光/遮挡，客户确认防止错误字段进入槽位，高风险或不确定情况转人工。视觉结果只是候选证据，不能直接报价、适配或定责。
17. **40张预跑为什么不是准确率？** API成功只证明鉴权、解析和Schema通了，没有gold就不知道字段对错。A/B两位审核员盲审，合并脚本找冲突，第三人裁决并通过许可/隐私门禁后，才能计算字段precision/recall/F1和拒识矩阵。
18. **Streamlit、FastAPI、LangSmith分别做什么？** Streamlit负责当前演示UI、调试和人工工作台；FastAPI用于外部API、鉴权和多客户端；LangSmith用于模型/链路Trace和评测实验。它们不是三选一，当前只实现了Streamlit和本地可观测。
19. **为什么后置Skills和multi-agent？** 当前单主图加确定性工具已经能清楚控制业务。只有稳定流程需要复用时封装Skill，复杂诊断需要权限/上下文隔离时增加sub-agent，并行角色收益超过通信和评测成本时才拆multi-agent。
20. **最诚实的生产边界是什么？** 当前是可演示、可离线评测、可解释的单实例MVP，不是多租户生产系统。公开图片没有完成双人gold，SQLite不适合多实例永久存储，网页搜索/FastAPI/LangSmith/分布式Harness尚未实现，库存和价格仍是模拟数据。

**回答结构：** 业务问题 -> 设计选择 -> 代码位置 -> 测试证据 -> 当前边界 -> 下一步。